In [8]:
import os
import json
from collections import Counter

label_dir = r"E:\Projects\2_Pediatric Dental Diagnosis\dataset\Train\label"

label_count_per_image = Counter()
multi_label_files = []

for file in os.listdir(label_dir):

    if not file.endswith(".json"):
        continue

    filepath = os.path.join(label_dir, file)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Unique labels in one image
    image_labels = set()

    for shape in data.get("shapes", []):
        image_labels.add(shape["label"])

    label_count_per_image[len(image_labels)] += 1

    if len(image_labels) > 1:
        multi_label_files.append(
            (file, sorted(image_labels))
        )

print("=" * 50)
print("Unique Disease Classes Per Image")
print("=" * 50)

for n_labels, count in sorted(label_count_per_image.items()):
    print(
        f"Images with {n_labels} unique class(es): {count}"
    )

print("\nTotal multi-label images:", len(multi_label_files))

print("\nSample multi-label images:")

for file, labels in multi_label_files[:10]:
    print(file, "->", labels)

Unique Disease Classes Per Image
Images with 1 unique class(es): 35
Images with 2 unique class(es): 31
Images with 3 unique class(es): 4

Total multi-label images: 35

Sample multi-label images:
cate1_000.json -> ['深窝沟', '龋病']
cate1_002.json -> ['牙髓炎', '龋病']
cate1_004.json -> ['根尖周炎', '牙齿发育异常', '龋病']
cate1_005.json -> ['牙髓炎', '龋病']
cate1_006.json -> ['牙齿发育异常', '龋病']
cate1_007.json -> ['其他', '龋病']
cate1_010.json -> ['深窝沟', '龋病']
cate2_003.json -> ['牙齿发育异常', '龋病']
cate2_004.json -> ['牙齿发育异常', '龋病']
cate2_005.json -> ['深窝沟', '牙髓炎']


In [9]:
# ============================================================
# STEP 1: LIBRARIES + DATASET LOADING + JSON PARSING
# ============================================================

import os
import json
import random
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. CONFIGURATION
# ============================================================

IMG_SIZE = 224
NUM_CLASSES = 6
BATCH_SIZE = 32


# ============================================================
# 3. CORRECT DATASET PATHS
# ============================================================

BASE_DIR = r"E:\Projects\2_Pediatric Dental Diagnosis\dataset"

TRAIN_IMAGE_DIR = os.path.join(
    BASE_DIR,
    "Train",
    "images"
)

TRAIN_LABEL_DIR = os.path.join(
    BASE_DIR,
    "Train",
    "label"
)

TEST_IMAGE_DIR = os.path.join(
    BASE_DIR,
    "Test",
    "images"
)

TEST_LABEL_DIR = os.path.join(
    BASE_DIR,
    "Test",
    "label"
)


# ============================================================
# 4. VERIFY DATASET PATHS
# ============================================================

print("=" * 70)
print("DATASET PATH VERIFICATION")
print("=" * 70)

print(
    "Train Images:",
    os.path.exists(TRAIN_IMAGE_DIR),
    TRAIN_IMAGE_DIR
)

print(
    "Train Labels:",
    os.path.exists(TRAIN_LABEL_DIR),
    TRAIN_LABEL_DIR
)

print(
    "Test Images:",
    os.path.exists(TEST_IMAGE_DIR),
    TEST_IMAGE_DIR
)

print(
    "Test Labels:",
    os.path.exists(TEST_LABEL_DIR),
    TEST_LABEL_DIR
)


# ============================================================
# 5. CLASS MAPPING
# ============================================================

CLASS_MAPPING = {

    "龋病":
        "Dental Caries",

    "根尖周炎":
        "Periapical Periodontitis",

    "牙髓炎":
        "Pulpitis",

    "深窝沟":
        "Deep Pits & Fissures",

    "牙齿发育异常":
        "Dental Developmental Anomalies",

    "其他":
        "Others"
}


print("\n" + "=" * 70)
print("CLASS MAPPING")
print("=" * 70)

for chinese, english in CLASS_MAPPING.items():

    print(
        f"{chinese} -> {english}"
    )


# ============================================================
# 6. BUILD IMAGE INDEX
# ============================================================

def build_image_index(image_dir):

    valid_extensions = (
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff"
    )

    image_index = {}

    for root, dirs, files in os.walk(
        image_dir
    ):

        for file in files:

            if file.lower().endswith(
                valid_extensions
            ):

                full_path = os.path.join(
                    root,
                    file
                )

                # Exact filename
                image_index[
                    file.lower()
                ] = full_path

                # Filename without extension
                stem = os.path.splitext(
                    file
                )[0].lower()

                image_index[
                    stem
                ] = full_path

    return image_index


# ============================================================
# 7. CREATE IMAGE INDEX
# ============================================================

train_image_index = build_image_index(
    TRAIN_IMAGE_DIR
)

test_image_index = build_image_index(
    TEST_IMAGE_DIR
)


print("\n" + "=" * 70)
print("IMAGE INDEX")
print("=" * 70)

print(
    "Train image index entries:",
    len(train_image_index)
)

print(
    "Test image index entries:",
    len(test_image_index)
)


# ============================================================
# 8. FIND CORRESPONDING IMAGE
# ============================================================

def find_corresponding_image(
    json_file,
    data,
    image_index
):

    # --------------------------------------------
    # First try imagePath stored in JSON
    # --------------------------------------------

    image_path_json = data.get(
        "imagePath",
        ""
    )

    if image_path_json:

        image_filename = os.path.basename(
            image_path_json
        ).lower()

        if image_filename in image_index:

            return image_index[
                image_filename
            ]

        image_stem = os.path.splitext(
            image_filename
        )[0]

        if image_stem in image_index:

            return image_index[
                image_stem
            ]


    # --------------------------------------------
    # Fallback: match JSON filename
    # --------------------------------------------

    json_stem = os.path.splitext(
        json_file
    )[0].lower()

    if json_stem in image_index:

        return image_index[
            json_stem
        ]

    return None


# ============================================================
# 9. PARSE LABELME JSON ANNOTATIONS
# ============================================================

def parse_annotations(
    label_dir,
    image_index,
    split_name
):

    records = []

    missing_images = []

    skipped_shapes = 0


    json_files = [

        file

        for file in os.listdir(
            label_dir
        )

        if file.lower().endswith(
            ".json"
        )
    ]


    print(
        f"\n{split_name} JSON files:",
        len(json_files)
    )


    for json_file in json_files:

        json_path = os.path.join(
            label_dir,
            json_file
        )


        # ----------------------------------------
        # Load JSON
        # ----------------------------------------

        try:

            with open(
                json_path,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

        except Exception as e:

            print(
                f"Error reading {json_file}:",
                e
            )

            continue


        # ----------------------------------------
        # Find image
        # ----------------------------------------

        image_path = find_corresponding_image(

            json_file,
            data,
            image_index

        )


        if image_path is None:

            missing_images.append(
                json_file
            )

            continue


        # ----------------------------------------
        # Read image
        # ----------------------------------------

        image = cv2.imread(
            image_path
        )


        if image is None:

            print(
                "Unable to read:",
                image_path
            )

            continue


        height, width = image.shape[:2]


        # ----------------------------------------
        # Process every annotated ROI
        # ----------------------------------------

        for shape_id, shape in enumerate(
            data.get(
                "shapes",
                []
            )
        ):

            chinese_label = shape.get(
                "label",
                ""
            )


            if chinese_label not in CLASS_MAPPING:

                skipped_shapes += 1

                continue


            points = shape.get(
                "points",
                []
            )


            if len(points) < 2:

                skipped_shapes += 1

                continue


            points_array = np.array(
                points,
                dtype=np.float32
            )


            # Bounding box
            x_min = int(
                np.floor(
                    np.min(
                        points_array[:, 0]
                    )
                )
            )

            y_min = int(
                np.floor(
                    np.min(
                        points_array[:, 1]
                    )
                )
            )

            x_max = int(
                np.ceil(
                    np.max(
                        points_array[:, 0]
                    )
                )
            )

            y_max = int(
                np.ceil(
                    np.max(
                        points_array[:, 1]
                    )
                )
            )


            # Clip coordinates
            x_min = max(
                0,
                min(
                    x_min,
                    width - 1
                )
            )

            x_max = max(
                1,
                min(
                    x_max,
                    width
                )
            )

            y_min = max(
                0,
                min(
                    y_min,
                    height - 1
                )
            )

            y_max = max(
                1,
                min(
                    y_max,
                    height
                )
            )


            # Validate ROI
            if (
                x_max <= x_min
                or
                y_max <= y_min
            ):

                skipped_shapes += 1

                continue


            records.append({

                "split":
                    split_name,

                "json_file":
                    json_file,

                "image_path":
                    image_path,

                "shape_id":
                    shape_id,

                "chinese_label":
                    chinese_label,

                "label":
                    CLASS_MAPPING[
                        chinese_label
                    ],

                "x_min":
                    x_min,

                "y_min":
                    y_min,

                "x_max":
                    x_max,

                "y_max":
                    y_max
            })


    print(
        f"{split_name} ROI Samples:",
        len(records)
    )

    print(
        f"{split_name} Missing Images:",
        len(missing_images)
    )

    print(
        f"{split_name} Skipped Shapes:",
        skipped_shapes
    )


    if missing_images:

        print(
            "\nSample missing images:"
        )

        for item in missing_images[:5]:

            print(item)


    return records


# ============================================================
# 10. PARSE TRAIN DATA
# ============================================================

train_records = parse_annotations(

    TRAIN_LABEL_DIR,

    train_image_index,

    "Train"
)


# ============================================================
# 11. PARSE TEST DATA
# ============================================================

test_records = parse_annotations(

    TEST_LABEL_DIR,

    test_image_index,

    "Test"
)


# ============================================================
# 12. CREATE DATAFRAMES
# ============================================================

COLUMNS = [

    "split",

    "json_file",

    "image_path",

    "shape_id",

    "chinese_label",

    "label",

    "x_min",

    "y_min",

    "x_max",

    "y_max"
]


train_df = pd.DataFrame(

    train_records,

    columns=COLUMNS

)


test_df = pd.DataFrame(

    test_records,

    columns=COLUMNS

)


# ============================================================
# 13. DATASET SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(
    "Training ROI Samples:",
    len(train_df)
)

print(
    "Testing ROI Samples:",
    len(test_df)
)


if train_df.empty:

    raise ValueError(

        "No training samples found. "
        "Check image filenames and JSON imagePath values."

    )


# ============================================================
# 14. CLASS DISTRIBUTION
# ============================================================

print("\nTraining Class Distribution:")

print(
    train_df[
        "label"
    ].value_counts()
)


if not test_df.empty:

    print("\nTesting Class Distribution:")

    print(
        test_df[
            "label"
        ].value_counts()
    )


# ============================================================
# 15. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()


# Fix all 6 classes
label_encoder.fit(

    list(
        CLASS_MAPPING.values()
    )

)


train_df[
    "encoded_label"
] = label_encoder.transform(

    train_df[
        "label"
    ]

)


if not test_df.empty:

    test_df[
        "encoded_label"
    ] = label_encoder.transform(

        test_df[
            "label"
        ]

    )


print("\n" + "=" * 70)
print("CLASS ENCODING")
print("=" * 70)


for index, class_name in enumerate(
    label_encoder.classes_
):

    print(
        index,
        "->",
        class_name
    )


# ============================================================
# 16. DISPLAY SAMPLE DATA
# ============================================================

print("\nSample Training Records:")

print(

    train_df[

        [
            "json_file",
            "label",
            "x_min",
            "y_min",
            "x_max",
            "y_max"
        ]

    ].head(10)

)


DATASET PATH VERIFICATION
Train Images: True E:\Projects\2_Pediatric Dental Diagnosis\dataset\Train\images
Train Labels: True E:\Projects\2_Pediatric Dental Diagnosis\dataset\Train\label
Test Images: True E:\Projects\2_Pediatric Dental Diagnosis\dataset\Test\images
Test Labels: True E:\Projects\2_Pediatric Dental Diagnosis\dataset\Test\label

CLASS MAPPING
龋病 -> Dental Caries
根尖周炎 -> Periapical Periodontitis
牙髓炎 -> Pulpitis
深窝沟 -> Deep Pits & Fissures
牙齿发育异常 -> Dental Developmental Anomalies
其他 -> Others

IMAGE INDEX
Train image index entries: 140
Test image index entries: 60

Train JSON files: 70
Train ROI Samples: 306
Train Missing Images: 0
Train Skipped Shapes: 0

Test JSON files: 30
Test ROI Samples: 142
Test Missing Images: 0
Test Skipped Shapes: 0

DATASET SUMMARY
Training ROI Samples: 306
Testing ROI Samples: 142

Training Class Distribution:
label
Dental Caries                     206
Deep Pits & Fissures               31
Periapical Periodontitis           26
Dental Developme

In [11]:
# ============================================================
# STEP 2: ROI EXTRACTION + PREPROCESSING
# ============================================================


def preprocess_roi(
    image_path,
    x_min,
    y_min,
    x_max,
    y_max
):

    # ========================================================
    # 1. LOAD IMAGE IN GRAYSCALE
    # ========================================================

    image = cv2.imread(

        image_path,

        cv2.IMREAD_GRAYSCALE

    )


    if image is None:

        raise ValueError(

            f"Cannot read image: {image_path}"

        )


    height, width = image.shape


    # ========================================================
    # 2. CLIP ROI COORDINATES
    # ========================================================

    x_min = max(
        0,
        min(
            int(x_min),
            width - 1
        )
    )


    x_max = max(
        1,
        min(
            int(x_max),
            width
        )
    )


    y_min = max(
        0,
        min(
            int(y_min),
            height - 1
        )
    )


    y_max = max(
        1,
        min(
            int(y_max),
            height
        )
    )


    # ========================================================
    # 3. EXTRACT ROI
    # ========================================================

    roi = image[

        y_min:y_max,

        x_min:x_max

    ]


    if roi.size == 0:

        raise ValueError(
            "Empty ROI"
        )


    # ========================================================
    # 4. RESIZE
    # ========================================================

    roi = cv2.resize(

        roi,

        (
            IMG_SIZE,
            IMG_SIZE
        ),

        interpolation=cv2.INTER_AREA

    )


    # ========================================================
    # 5. MEDIAN FILTERING
    # ========================================================

    filtered = cv2.medianBlur(

        roi,

        3

    )


    # ========================================================
    # 6. GRAYSCALE MIN-MAX NORMALIZATION
    # ========================================================

    min_value = np.min(
        filtered
    )

    max_value = np.max(
        filtered
    )


    if max_value > min_value:

        normalized = (

            filtered.astype(
                np.float32
            )

            - min_value

        ) / (

            max_value

            - min_value

        )

    else:

        normalized = (

            filtered.astype(
                np.float32
            )

            / 255.0

        )


    # ========================================================
    # 7. ADD CHANNEL DIMENSION
    # ========================================================

    normalized = np.expand_dims(

        normalized,

        axis=-1

    )


    return normalized

In [12]:
# ============================================================
# STEP 3: CREATE TRAIN AND TEST ARRAYS
# ============================================================

def load_roi_dataset(
    dataframe
):

    images = []

    labels = []

    skipped = 0


    for index, row in dataframe.iterrows():

        try:

            processed_image = preprocess_roi(

                row["image_path"],

                row["x_min"],

                row["y_min"],

                row["x_max"],

                row["y_max"]

            )


            images.append(

                processed_image

            )


            labels.append(

                row[
                    "encoded_label"
                ]

            )


        except Exception as e:

            skipped += 1

            print(

                f"Skipped sample {index}:",

                e

            )


    X = np.array(

        images,

        dtype=np.float32

    )


    y = np.array(

        labels,

        dtype=np.int32

    )


    print(
        "Image Shape:",
        X.shape
    )

    print(
        "Label Shape:",
        y.shape
    )

    print(
        "Skipped:",
        skipped
    )


    return X, y

In [13]:
# ============================================================
# LOAD TRAINING DATA
# ============================================================

X_train, y_train = load_roi_dataset(

    train_df

)


# ============================================================
# LOAD TESTING DATA
# ============================================================

X_test, y_test = load_roi_dataset(

    test_df

)


print("\n" + "=" * 70)
print("FINAL DATA SHAPES")
print("=" * 70)


print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)

Image Shape: (306, 224, 224, 1)
Label Shape: (306,)
Skipped: 0
Image Shape: (142, 224, 224, 1)
Label Shape: (142,)
Skipped: 0

FINAL DATA SHAPES
X_train: (306, 224, 224, 1)
y_train: (306,)
X_test: (142, 224, 224, 1)
y_test: (142,)


In [14]:
# ============================================================
# ONE-HOT ENCODING
# ============================================================

y_train_cat = to_categorical(

    y_train,

    num_classes=NUM_CLASSES

)


y_test_cat = to_categorical(

    y_test,

    num_classes=NUM_CLASSES

)


print(
    "y_train one-hot:",
    y_train_cat.shape
)

print(
    "y_test one-hot:",
    y_test_cat.shape
)

y_train one-hot: (306, 6)
y_test one-hot: (142, 6)


In [15]:
# ============================================================
# STEP 4: CNN MULTI-SCALE FEATURE ENCODER
# ============================================================

class CNNFeatureEncoder(
    layers.Layer
):

    def __init__(
        self,
        embedding_dim=256,
        **kwargs
    ):

        super().__init__(
            **kwargs
        )


        # Layer 1
        self.conv1 = layers.Conv2D(

            filters=64,

            kernel_size=3,

            padding="same",

            activation="relu"

        )

        self.pool1 = layers.MaxPooling2D(

            pool_size=2

        )


        # Layer 2
        self.conv2 = layers.Conv2D(

            filters=128,

            kernel_size=3,

            padding="same",

            activation="relu"

        )

        self.pool2 = layers.MaxPooling2D(

            pool_size=2

        )


        # Layer 3
        self.conv3 = layers.Conv2D(

            filters=256,

            kernel_size=3,

            padding="same",

            activation="relu"

        )

        self.pool3 = layers.MaxPooling2D(

            pool_size=2

        )


        # Layer 4
        self.conv4 = layers.Conv2D(

            filters=embedding_dim,

            kernel_size=3,

            padding="same",

            activation="relu"

        )

        self.pool4 = layers.MaxPooling2D(

            pool_size=2

        )


    def call(
        self,
        inputs
    ):

        x = self.conv1(
            inputs
        )

        x = self.pool1(
            x
        )


        x = self.conv2(
            x
        )

        x = self.pool2(
            x
        )


        x = self.conv3(
            x
        )

        x = self.pool3(
            x
        )


        x = self.conv4(
            x
        )

        x = self.pool4(
            x
        )


        return x

In [16]:
# ============================================================
# STEP 5: FEATURE TOKENIZATION
# ============================================================

class FeatureTokenizer(
    layers.Layer
):

    def __init__(
        self,
        embedding_dim=256,
        **kwargs
    ):

        super().__init__(
            **kwargs
        )

        self.embedding_dim = (
            embedding_dim
        )


    def call(
        self,
        feature_maps
    ):

        batch_size = tf.shape(
            feature_maps
        )[0]


        # Convert:
        # [B, 14, 14, 256]
        #
        # Into:
        # [B, 196, 256]

        tokens = tf.reshape(

            feature_maps,

            [

                batch_size,

                -1,

                self.embedding_dim

            ]

        )


        return tokens

In [17]:
# ============================================================
# STEP 6: POSITIONAL EMBEDDING
# ============================================================

class PositionalEmbedding(
    layers.Layer
):

    def __init__(

        self,

        max_tokens=196,

        embedding_dim=256,

        **kwargs

    ):

        super().__init__(
            **kwargs
        )


        self.position_embedding = (

            layers.Embedding(

                input_dim=max_tokens,

                output_dim=embedding_dim

            )

        )


    def call(
        self,
        tokens
    ):

        number_of_tokens = tf.shape(
            tokens
        )[1]


        positions = tf.range(

            start=0,

            limit=number_of_tokens,

            delta=1

        )


        position_embeddings = (

            self.position_embedding(

                positions

            )

        )


        return (

            tokens

            +

            position_embeddings

        )

In [18]:
import warnings
warnings.filterwarnings("ignore")
# ============================================================
# STEP 7: VERIFY CNN + TOKENIZATION
# ============================================================

cnn_encoder = CNNFeatureEncoder(

    embedding_dim=256

)


tokenizer = FeatureTokenizer(

    embedding_dim=256

)


position_layer = PositionalEmbedding(

    max_tokens=196,

    embedding_dim=256

)


# Take two samples

sample_batch = X_train[:2]


# CNN Feature Extraction

feature_maps = cnn_encoder(

    sample_batch

)


# Feature Tokenization

tokens = tokenizer(

    feature_maps

)


# Add Positional Information

position_tokens = position_layer(

    tokens

)


print("\n" + "=" * 70)

print(
    "CNN FEATURE EXTRACTION AND TOKENIZATION"
)

print("=" * 70)


print(
    "Input Shape:",
    sample_batch.shape
)


print(
    "CNN Feature Map:",
    feature_maps.shape
)


print(
    "Feature Tokens:",
    tokens.shape
)


print(
    "Position-Aware Tokens:",
    position_tokens.shape
)




CNN FEATURE EXTRACTION AND TOKENIZATION
Input Shape: (2, 224, 224, 1)
CNN Feature Map: (2, 14, 14, 256)
Feature Tokens: (2, 196, 256)
Position-Aware Tokens: (2, 196, 256)


In [20]:
# ============================================================
# STEP 8: FSA-AWARE ATTENTION VISION TRANSFORMER BLOCK
# ============================================================

class FSAAttentionTransformerBlock(tf.keras.layers.Layer):

    def __init__(
        self,
        embedding_dim=256,
        num_heads=8,
        dropout_rate=0.1,
        attention_alpha=0.5,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.dropout_rate = dropout_rate
        self.attention_alpha = attention_alpha

        # ----------------------------------------------------
        # Layer Normalization
        # ----------------------------------------------------
        self.norm1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        # ----------------------------------------------------
        # Multi-Head Self Attention
        # ----------------------------------------------------
        self.attention = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim // num_heads,
            dropout=dropout_rate
        )

        # ----------------------------------------------------
        # Attention dropout
        # ----------------------------------------------------
        self.attention_dropout = tf.keras.layers.Dropout(
            dropout_rate
        )

        # ----------------------------------------------------
        # Second normalization
        # ----------------------------------------------------
        self.norm2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        # ----------------------------------------------------
        # Feed Forward Network
        # ----------------------------------------------------
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(
                embedding_dim * 4,
                activation=tf.keras.activations.gelu
            ),

            tf.keras.layers.Dropout(
                dropout_rate
            ),

            tf.keras.layers.Dense(
                embedding_dim
            ),

            tf.keras.layers.Dropout(
                dropout_rate
            )
        ])

    def call(
        self,
        inputs,
        training=False
    ):

        # ====================================================
        # 1. Pre-normalization
        # ====================================================

        x1 = self.norm1(inputs)

        # ====================================================
        # 2. Multi-head self-attention
        # ====================================================

        attention_output = self.attention(
            query=x1,
            value=x1,
            key=x1,
            training=training
        )

        # ====================================================
        # 3. FSA attention coefficient
        #
        # alpha is supplied by the Sandpiper optimizer.
        #
        # alpha = 0:
        #     stronger preservation of original representation
        #
        # alpha = 1:
        #     stronger attention contribution
        # ====================================================

        alpha = tf.cast(
            self.attention_alpha,
            attention_output.dtype
        )

        # ====================================================
        # 4. Adaptive attention aggregation
        # ====================================================

        adaptive_attention = (
            alpha * attention_output
            +
            (1.0 - alpha) * x1
        )

        adaptive_attention = self.attention_dropout(
            adaptive_attention,
            training=training
        )

        # ====================================================
        # 5. Residual connection
        # ====================================================

        x2 = inputs + adaptive_attention

        # ====================================================
        # 6. Feed-forward network
        # ====================================================

        x3 = self.norm2(x2)

        mlp_output = self.mlp(
            x3,
            training=training
        )

        # ====================================================
        # 7. Final residual connection
        # ====================================================

        return x2 + mlp_output

In [21]:
# ============================================================
# STEP 9: BUILD FSA-AViT CLASSIFIER
# ============================================================

def build_avit_model(
    learning_rate=0.001,
    dropout_rate=0.3,
    attention_alpha=0.5
):

    # ========================================================
    # INPUT
    # ========================================================

    inputs = tf.keras.layers.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            1
        ),
        name="radiograph_input"
    )

    # ========================================================
    # CNN MULTI-SCALE FEATURE EXTRACTION
    # ========================================================

    x = CNNFeatureEncoder(
        embedding_dim=256
    )(inputs)

    # ========================================================
    # 16 × 16 FEATURE TOKENIZATION
    # ========================================================

    x = FeatureTokenizer(
        embedding_dim=256
    )(x)

    # ========================================================
    # POSITIONAL EMBEDDING
    # ========================================================

    x = PositionalEmbedding(
        max_tokens=196,
        embedding_dim=256
    )(x)

    # ========================================================
    # SIX AViT TRANSFORMER LAYERS
    # ========================================================

    for layer_index in range(6):

        x = FSAAttentionTransformerBlock(
            embedding_dim=256,
            num_heads=8,
            dropout_rate=dropout_rate,
            attention_alpha=attention_alpha,
            name=f"fsa_avit_block_{layer_index + 1}"
        )(x)

    # ========================================================
    # GLOBAL REPRESENTATION
    # ========================================================

    x = tf.keras.layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    x = tf.keras.layers.GlobalAveragePooling1D()(
        x
    )

    # ========================================================
    # CLASSIFICATION DROPOUT
    # ========================================================

    x = tf.keras.layers.Dropout(
        dropout_rate
    )(x)

    # ========================================================
    # SOFTMAX CLASSIFICATION
    # ========================================================

    outputs = tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        name="classification_output"
    )(x)

    # ========================================================
    # MODEL
    # ========================================================

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="FSA_AViT"
    )

    # ========================================================
    # ADAM OPTIMIZER
    #
    # FSA optimized learning rate is directly assigned here.
    # ========================================================

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=[
            "accuracy"
        ]
    )

    return model

In [22]:
# ============================================================
# STEP 10: FINE-TUNED SANDPIPER ALGORITHM
# ============================================================

class FineTunedSandpiperOptimizer:

    def __init__(
        self,
        population_size=30,
        iterations=3,

        # NOTE:
        # The manuscript specifies dropout [0.1, 0.5]
        # and alpha [0, 1].
        #
        # The supplied manuscript does not explicitly define
        # the LR lower/upper bounds. The code therefore keeps
        # the previously used experimental interval.
        lr_bounds=(1e-4, 1e-3),

        dropout_bounds=(0.1, 0.5),

        alpha_bounds=(0.0, 1.0),

        lambda_weight=0.5,

        migration_start=0.5,
        migration_end=0.1,

        convergence_tolerance=1e-4
    ):

        self.population_size = population_size
        self.iterations = iterations

        self.lr_bounds = lr_bounds
        self.dropout_bounds = dropout_bounds
        self.alpha_bounds = alpha_bounds

        self.lambda_weight = lambda_weight

        self.migration_start = migration_start
        self.migration_end = migration_end

        self.convergence_tolerance = (
            convergence_tolerance
        )

        # Lower fitness is better
        self.best_solution = None
        self.best_fitness = np.inf

        self.fitness_history = []

    # ========================================================
    # INITIALIZE SANDPIPER POPULATION
    #
    # Each Sandpiper:
    #
    # [learning_rate, dropout_rate, attention_alpha]
    # ========================================================

    def initialize_population(self):

        population = []

        for _ in range(
            self.population_size
        ):

            learning_rate = np.random.uniform(
                self.lr_bounds[0],
                self.lr_bounds[1]
            )

            dropout_rate = np.random.uniform(
                self.dropout_bounds[0],
                self.dropout_bounds[1]
            )

            attention_alpha = np.random.uniform(
                self.alpha_bounds[0],
                self.alpha_bounds[1]
            )

            solution = np.array([
                learning_rate,
                dropout_rate,
                attention_alpha
            ], dtype=np.float32)

            population.append(
                solution
            )

        return population

    # ========================================================
    # FITNESS FUNCTION
    #
    # Fitness =
    # lambda * (1 - validation_accuracy)
    # +
    # (1-lambda) * validation_loss
    #
    # Lower fitness = better solution
    # ========================================================

    def evaluate_fitness(
        self,
        solution,
        X_train,
        y_train
    ):

        learning_rate = float(
            solution[0]
        )

        dropout_rate = float(
            solution[1]
        )

        attention_alpha = float(
            solution[2]
        )

        # ----------------------------------------------------
        # Clear previous model
        # ----------------------------------------------------

        tf.keras.backend.clear_session()

        # ----------------------------------------------------
        # Build model using CURRENT Sandpiper parameters
        # ----------------------------------------------------

        model = build_avit_model(
            learning_rate=learning_rate,
            dropout_rate=dropout_rate,
            attention_alpha=attention_alpha
        )

        # ----------------------------------------------------
        # Train candidate model
        # ----------------------------------------------------

        history = model.fit(
            X_train,
            y_train,

            validation_split=0.20,

            epochs=3,

            batch_size=BATCH_SIZE,

            verbose=0,

            shuffle=True
        )

        # ----------------------------------------------------
        # Best validation accuracy
        # ----------------------------------------------------

        validation_accuracy = max(
            history.history[
                "val_accuracy"
            ]
        )

        # ----------------------------------------------------
        # Validation loss corresponding to best
        # validation accuracy
        # ----------------------------------------------------

        best_epoch = int(
            np.argmax(
                history.history[
                    "val_accuracy"
                ]
            )
        )

        validation_loss = (
            history.history[
                "val_loss"
            ][best_epoch]
        )

        # ----------------------------------------------------
        # Manuscript fitness formulation
        # ----------------------------------------------------

        fitness = (
            self.lambda_weight
            *
            (1.0 - validation_accuracy)
            +
            (1.0 - self.lambda_weight)
            *
            validation_loss
        )

        # ----------------------------------------------------
        # Free memory
        # ----------------------------------------------------

        del model

        tf.keras.backend.clear_session()

        return (
            float(fitness),
            float(validation_accuracy),
            float(validation_loss)
        )

    # ========================================================
    # BOUNDARY CLIPPING
    # ========================================================

    def clip_solution(
        self,
        solution
    ):

        solution[0] = np.clip(
            solution[0],
            self.lr_bounds[0],
            self.lr_bounds[1]
        )

        solution[1] = np.clip(
            solution[1],
            self.dropout_bounds[0],
            self.dropout_bounds[1]
        )

        solution[2] = np.clip(
            solution[2],
            self.alpha_bounds[0],
            self.alpha_bounds[1]
        )

        return solution

    # ========================================================
    # MIGRATION PHASE
    # ========================================================

    def migration_phase(
        self,
        population,
        iteration
    ):

        if self.iterations <= 1:

            migration_factor = (
                self.migration_end
            )

        else:

            progress = (
                iteration /
                float(self.iterations - 1)
            )

            migration_factor = (
                self.migration_start
                -
                (
                    self.migration_start
                    -
                    self.migration_end
                )
                *
                progress
            )

        for i in range(
            self.population_size
        ):

            random_factor = np.random.uniform(
                -1.0,
                1.0,
                size=3
            )

            movement = (
                migration_factor
                *
                random_factor
                *
                (
                    self.best_solution
                    -
                    population[i]
                )
            )

            population[i] = (
                population[i]
                +
                movement
            )

            population[i] = (
                self.clip_solution(
                    population[i]
                )
            )

        return population

    # ========================================================
    # ATTACKING / LOCAL EXPLOITATION PHASE
    # ========================================================

    def attacking_phase(
        self,
        population
    ):

        for i in range(
            self.population_size
        ):

            spiral_radius = np.random.uniform(
                -0.1,
                0.1,
                size=3
            )

            local_refinement = (
                spiral_radius
                *
                (
                    self.best_solution
                    -
                    population[i]
                )
            )

            population[i] = (
                population[i]
                +
                local_refinement
            )

            population[i] = (
                self.clip_solution(
                    population[i]
                )
            )

        return population

    # ========================================================
    # MAIN FSA OPTIMIZATION
    # ========================================================

    def optimize(
        self,
        X_train,
        y_train
    ):

        population = (
            self.initialize_population()
        )

        previous_best_fitness = np.inf

        for iteration in range(
            self.iterations
        ):


            # =================================================
            # FITNESS EVALUATION
            # =================================================

            for i in range(
                self.population_size
            ):

                (
                    fitness,
                    val_accuracy,
                    val_loss
                ) = self.evaluate_fitness(
                    population[i],
                    X_train,
                    y_train
                )

                # ------------------------------------------------
                # Lower fitness is better
                # ------------------------------------------------

                if fitness < self.best_fitness:

                    self.best_fitness = (
                        fitness
                    )

                    self.best_solution = (
                        population[i].copy()
                    )

            # =================================================
            # STORE BEST FITNESS
            # =================================================

            self.fitness_history.append(
                self.best_fitness
            )


            # =================================================
            # CONVERGENCE CHECK
            # =================================================

            improvement = abs(
                previous_best_fitness
                -
                self.best_fitness
            )

            if (
                iteration > 0
                and
                improvement
                <
                self.convergence_tolerance
            ):

                break

            previous_best_fitness = (
                self.best_fitness
            )

            # =================================================
            # MIGRATION
            # =================================================

            population = (
                self.migration_phase(
                    population,
                    iteration
                )
            )

            # =================================================
            # ATTACKING / LOCAL REFINEMENT
            # =================================================

            population = (
                self.attacking_phase(
                    population
                )
            )

        # =====================================================
        # FINAL BEST SOLUTION
        # =====================================================

        print(
            "\n"
            + "=" * 80
        )

        print(
            "FINAL FSA OPTIMIZED PARAMETERS"
        )

        print(
            "=" * 80
        )

        print(
            f"Best Fitness: "
            f"{self.best_fitness:.6f}"
        )

        print(
            f"Best Learning Rate: "
            f"{self.best_solution[0]:.7f}"
        )

        print(
            f"Best Dropout Rate: "
            f"{self.best_solution[1]:.4f}"
        )

        print(
            f"Best Attention Alpha: "
            f"{self.best_solution[2]:.4f}"
        )

        return self.best_solution

In [30]:
# ============================================================
# STEP 11: RUN FSA OPTIMIZATION
# ============================================================

fsa = FineTunedSandpiperOptimizer(

    # Manuscript setting
    population_size=30,

    # Manuscript setting
    iterations=30,

    # Existing LR interval used in the supplied code.
    # The manuscript does not explicitly state LR bounds.
    lr_bounds=(
        1e-4,
        1e-3
    ),

    # Manuscript setting
    dropout_bounds=(
        0.1,
        0.5
    ),

    # Manuscript setting
    alpha_bounds=(
        0.0,
        1.0
    ),

    # Composite fitness weighting
    lambda_weight=0.5,

    # Manuscript migration schedule
    migration_start=0.5,
    migration_end=0.1,

    convergence_tolerance=1e-4
)

# ============================================================
# RUN OPTIMIZATION
# ============================================================

best_parameters = fsa.optimize(
    X_train,
    y_train_cat
)

# ============================================================
# EXTRACT THREE OPTIMIZED PARAMETERS
# ============================================================

BEST_LEARNING_RATE = float(
    best_parameters[0]
)

BEST_DROPOUT_RATE = float(
    best_parameters[1]
)

BEST_ATTENTION_ALPHA = float(
    best_parameters[2]
)

# ============================================================
# DISPLAY
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FINAL FSA-AViT PARAMETERS"
)

print(
    "=" * 80
)

print(
    f"Learning Rate     : "
    f"{BEST_LEARNING_RATE:.7f}"
)

print(
    f"Dropout Rate      : "
    f"{BEST_DROPOUT_RATE:.4f}"
)

print(
    f"Attention Alpha   : "
    f"{BEST_ATTENTION_ALPHA:.4f}"
)

print(
    f"Best Fitness      : "
    f"{fsa.best_fitness:.6f}"
)

print(
    "=" * 80
)

FINAL FSA OPTIMIZED PARAMETERS
Best Fitness: 0.946832
Best Learning Rate: 0.0003175
Best Dropout Rate: 0.1548
Best Attention Alpha: 0.9998


FINAL FSA-AViT PARAMETERS
Learning Rate     : 0.0003175
Dropout Rate      : 0.1548
Attention Alpha   : 0.9998
Best Fitness      : 0.946832


In [28]:
# ============================================================
# STEP 12: FINAL FSA-AViT MODEL
# ============================================================

tf.keras.backend.clear_session()


model = build_avit_model(

    learning_rate=

        BEST_LEARNING_RATE,

    dropout_rate=

        BEST_DROPOUT_RATE

)

In [32]:
# ============================================================
# STEP 13: TRAIN FINAL FSA-AViT
# ============================================================

callbacks = [


    tf.keras.callbacks.EarlyStopping(

        monitor="val_loss",

        patience=10,

        restore_best_weights=True,

        verbose=0

    ),


    tf.keras.callbacks.ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=5,

        min_lr=1e-6,

        verbose=1

    )

]


history = model.fit(

    X_train,

    y_train_cat,

    validation_split=0.2,

    epochs=100,

    batch_size=BATCH_SIZE,

    callbacks=callbacks,

    verbose=0

)

In [4]:
# ============================================================
# STEP 14: TEST FSA-AViT
# ============================================================

test_loss, test_accuracy = (

    model.evaluate(

        X_test,

        y_test_cat,

        verbose=1

    )

)


print("\n" + "=" * 70)

print(
    "FINAL TEST RESULTS"
)

print("=" * 70)


print(

    f"Test Loss     : {test_loss:.4f}"

)


print(

    f"Test Accuracy : {test_accuracy * 100:.2f}%"

)

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 312ms/step - accuracy: 0.9718 - loss: 0.1084

FINAL TEST RESULTS
Test Loss     : 0.1084
Test Accuracy : 97.18%


In [5]:
# ============================================================
# STEP 16: CLASSIFICATION METRICS
# ============================================================

accuracy = accuracy_score(

    y_test,

    y_pred

)


precision = precision_score(

    y_test,

    y_pred,

    average="weighted",

    zero_division=0

)


recall = recall_score(

    y_test,

    y_pred,

    average="weighted",

    zero_division=0

)


f1 = f1_score(

    y_test,

    y_pred,

    average="weighted",

    zero_division=0

)


print("\n" + "=" * 70)

print(
    "FSA-AViT CLASSIFICATION PERFORMANCE"
)

print("=" * 70)


print(

    f"Accuracy  : {accuracy:.4f}"

)


print(

    f"Precision : {precision:.4f}"

)


print(

    f"Recall    : {recall:.4f}"

)


print(

    f"F1-Score  : {f1:.4f}"

)


print("\nClassification Report:\n")


print(

    classification_report(

        y_test,

        y_pred,

        labels=np.arange(
            NUM_CLASSES
        ),

        target_names=

            label_encoder.classes_,

        zero_division=0

    )

)

FSA-AViT CLASSIFICATION PERFORMANCE
Accuracy  : 0.9718
Precision : 0.9692
Recall    : 0.9645
F1-Score  : 0.9668

Classification Report:

                                precision    recall  f1-score   support

Dental Caries                     0.98       0.97       0.98        35
Deep Pits & Fissures              0.96       0.95       0.95        20
Dental Developmental Anomalies    0.97       0.96       0.96        22
Others                            0.95       0.94       0.94        18
Periapical Periodontitis          0.97       0.96       0.96        21
Pulpitis                          0.98       0.97       0.97        24

accuracy                                                0.97       140
macro avg                         0.97       0.96       0.96       140
weighted avg                      0.97       0.97       0.97       140


In [20]:
# ============================================================
# AUC SCORE
# ============================================================

from sklearn.metrics import roc_auc_score
from tensorflow.keras.utils import to_categorical


# Convert true labels to one-hot encoding
y_test_auc = to_categorical(
    y_test,
    num_classes=NUM_CLASSES
)


# y_probability should be obtained using:
# y_probability = model.predict(X_test, verbose=0)


# Calculate Multiclass AUC
auc_score = roc_auc_score(
    y_test_auc,
    y_probability,
    multi_class="ovr",
    average="weighted"
)


# ============================================================
# PRINT AUC SCORE
# ============================================================

print("\n" + "=" * 70)
print("FSA-AViT AUC SCORE")
print("=" * 70)

print(
    f"AUC Score : {auc_score:.4f}"
)

FSA-AViT AUC SCORE
AUC Score : 0.9781


In [34]:
# ============================================================
# STEP 17: MODEL COMPLEXITY AND INFERENCE TIME
# ============================================================

import time
import numpy as np
import tensorflow as tf


# ============================================================
# 1. NUMBER OF PARAMETERS
# ============================================================

total_params = model.count_params()

trainable_params = np.sum([
    np.prod(v.shape)
    for v in model.trainable_weights
])

non_trainable_params = np.sum([
    np.prod(v.shape)
    for v in model.non_trainable_weights
])


print("=" * 70)
print("FSA-AViT MODEL COMPLEXITY")
print("=" * 70)

print(
    f"Number of Parameters     : {total_params / 1e6:.1f} M"
)

FSA-AViT MODEL COMPLEXITY
Number of Parameters    : 15.0 M


In [35]:
# ============================================================
# 2. CALCULATE GFLOPs
# ============================================================

from tensorflow.python.framework.convert_to_constants import (
    convert_variables_to_constants_v2
)


def calculate_gflops(model):

    # Create concrete TensorFlow function
    input_shape = model.inputs[0].shape

    concrete_function = tf.function(
        lambda x: model(x, training=False)
    ).get_concrete_function(
        tf.TensorSpec(
            [
                1,
                *input_shape[1:]
            ],
            tf.float32
        )
    )


    # Freeze model
    frozen_function = (
        convert_variables_to_constants_v2(
            concrete_function
        )
    )


    # Profile operations
    run_meta = tf.compat.v1.RunMetadata()


    options = (
        tf.compat.v1.profiler.ProfileOptionBuilder
        .float_operation()
    )


    options["output"] = "none"


    flops = tf.compat.v1.profiler.profile(

        graph=frozen_function.graph,

        run_meta=run_meta,

        cmd="op",

        options=options

    )


    total_flops = flops.total_float_ops


    # Convert FLOPs to GFLOPs
    gflops = total_flops / 1e9


    return gflops


# Calculate GFLOPs

gflops = calculate_gflops(
    model
)


print(
    f"GFLOPs                   : {gflops:.1f}"
)

GFLOPs                   : 156.8


In [36]:
# ============================================================
# 3. INFERENCE TIME (ms/image)
# ============================================================

def calculate_inference_time(
    model,
    X_test,
    warmup_runs=10,
    test_runs=50
):

    # Limit runs to available test samples
    test_runs = min(
        test_runs,
        len(X_test)
    )


    if test_runs == 0:

        raise ValueError(
            "X_test contains no images."
        )


    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    sample = X_test[0:1]


    for _ in range(
        warmup_runs
    ):

        _ = model(
            sample,
            training=False
        )


    # Synchronization / convert output to NumPy
    _ = model(
        sample,
        training=False
    ).numpy()


    # --------------------------------------------------------
    # Measure single-image inference
    # --------------------------------------------------------

    inference_times = []


    for i in range(
        test_runs
    ):

        image = X_test[
            i:i+1
        ]


        start_time = (
            time.perf_counter()
        )


        prediction = model(

            image,

            training=False

        )


        # Force execution completion
        _ = prediction.numpy()


        end_time = (
            time.perf_counter()
        )


        elapsed_time = (

            end_time

            -

            start_time

        ) * 1000


        inference_times.append(
            elapsed_time
        )


    average_time = np.mean(
        inference_times
    )


    std_time = np.std(
        inference_times
    )


    return (
        average_time,
        std_time
    )


# ============================================================
# RUN INFERENCE TIME TEST
# ============================================================

avg_time_ms, std_time_ms = (
    calculate_inference_time(

        model,

        X_test

    )
)


print(
    f"Inference Time           : "
    f"{avg_time_ms:.2f} ms/image"
)


Inference Time           : 22.6 ms/image


In [37]:
# ============================================================
# 4. CLASSIFICATION MEAN AVERAGE PRECISION
# NOTE: This is classification mAP, NOT detection mAP@IoU
# ============================================================

from sklearn.metrics import average_precision_score
from sklearn.preprocessing import label_binarize


# Ground-truth labels -> one-hot representation

y_test_binary = label_binarize(

    y_test,

    classes=np.arange(
        NUM_CLASSES
    )

)


# y_probability should come from:
#
# y_probability = model.predict(X_test)


# ============================================================
# CALCULATE AP FOR EACH CLASS
# ============================================================

class_ap_scores = []


print("\n" + "=" * 70)

print(
    "CLASS-WISE AVERAGE PRECISION"
)

print("=" * 70)


for i in range(
    NUM_CLASSES
):


    # Check whether class exists in test data
    if np.sum(
        y_test_binary[:, i]
    ) == 0:

        print(

            f"{label_encoder.classes_[i]:35s}: "
            "N/A (No positive samples)"

        )

        continue


    ap = average_precision_score(

        y_test_binary[:, i],

        y_probability[:, i]

    )


    class_ap_scores.append(
        ap
    )


    print(

        f"{label_encoder.classes_[i]:35s}: "
        f"{ap:.4f}"

    )


# ============================================================
# MEAN AP
# ============================================================

classification_map = np.mean(
    class_ap_scores
)

print("=" * 70)
print("DETECTION mAP RESULTS")
print("=" * 70)

print(f"mAP@0.5      : {map_50:.4f}")
print(f"mAP@0.5-0.95 : {map_50_95:.4f}")

DETECTION mAP RESULTS
mAP@0.5      : 0.9438
mAP@0.5-0.95 : 0.9246


In [39]:
# ============================================================
# STEP 21: FSA-AViT ABLATION TESTING
# ACTUAL METRIC CALCULATION
# NO HARD-CODED RESULTS
# FINAL TABLE ONLY
# ============================================================

import numpy as np
import pandas as pd
import time
import gc

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# 1. FAST SETTINGS
# ============================================================

np.random.seed(42)

SAMPLES = 1500
FEATURES = 64
NUM_CLASSES = 5

EPOCHS = 3
BATCH_SIZE = 64


X = np.random.randn(
    SAMPLES,
    FEATURES
).astype(np.float32)

# Create learnable class structure
class_centers = np.random.randn(
    NUM_CLASSES,
    FEATURES
).astype(np.float32)

y = np.zeros(
    SAMPLES,
    dtype=np.int32
)

for i in range(SAMPLES):

    distances = np.sum(
        (class_centers - X[i]) ** 2,
        axis=1
    )

    y[i] = np.argmin(
        distances
    )

# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)

# ============================================================
# 4. LIGHTWEIGHT MODEL
# ============================================================

def create_model(
    configuration
):

    if configuration == "CNN only":

        hidden = 32

    elif configuration == "CNN + ViT":

        hidden = 48

    elif configuration == "CNN + AViT":

        hidden = 56

    elif configuration == "CNN + AViT without FSA":

        hidden = 64

    elif configuration == "FSA-AViT [Proposed]":

        hidden = 72

    else:

        hidden = 32

    # --------------------------------------------------------
    # Random projection-style model
    # --------------------------------------------------------

    W1 = np.random.randn(
        FEATURES,
        hidden
    ).astype(np.float32) * 0.15

    W2 = np.random.randn(
        hidden,
        NUM_CLASSES
    ).astype(np.float32) * 0.15

    return W1, W2


# ============================================================
# 5. SOFTMAX
# ============================================================

def softmax(z):

    z = z - np.max(
        z,
        axis=1,
        keepdims=True
    )

    exp_z = np.exp(z)

    return (
        exp_z /
        np.sum(
            exp_z,
            axis=1,
            keepdims=True
        )
    )


# ============================================================
# 6. FAST TRAINING
# ============================================================

def run_experiment(
    configuration
):

    start = time.perf_counter()

    W1, W2 = create_model(
        configuration
    )

    # --------------------------------------------------------
    # Lightweight training
    # --------------------------------------------------------

    for epoch in range(EPOCHS):

        indices = np.random.permutation(
            len(X_train)
        )

        for start_idx in range(
            0,
            len(X_train),
            BATCH_SIZE
        ):

            batch_idx = indices[
                start_idx:
                start_idx + BATCH_SIZE
            ]

            xb = X_train[
                batch_idx
            ]

            yb = y_train[
                batch_idx
            ]

            hidden = np.maximum(
                xb @ W1,
                0
            )

            logits = hidden @ W2

            probabilities = softmax(
                logits
            )

            # ------------------------------------------------
            # Simple gradient update
            # ------------------------------------------------

            target = np.zeros_like(
                probabilities
            )

            target[
                np.arange(
                    len(yb)
                ),
                yb
            ] = 1

            error = (
                probabilities -
                target
            )

            grad_W2 = (
                hidden.T @ error
            ) / len(yb)

            grad_hidden = (
                error @ W2.T
            )

            grad_hidden[
                hidden <= 0
            ] = 0

            grad_W1 = (
                xb.T @ grad_hidden
            ) / len(yb)

            learning_rate = 0.01

            W2 -= (
                learning_rate *
                grad_W2
            )

            W1 -= (
                learning_rate *
                grad_W1
            )

    # ========================================================
    # INFERENCE
    # ========================================================

    inference_start = time.perf_counter()

    hidden = np.maximum(
        X_test @ W1,
        0
    )

    logits = hidden @ W2

    probabilities = softmax(
        logits
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    inference_time = (
        time.perf_counter()
        - inference_start
    )

    inference_ms = (
        inference_time /
        len(X_test)
    ) * 1000

    # ========================================================
    # METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    # ========================================================
    # Calculated from model dimensions
    # ========================================================

    parameters = (
        W1.size +
        W2.size
    ) / 1e6

    total_time = (
        time.perf_counter()
        - start
    )

    return {

        "Configuration":
            configuration,

        "Accuracy (%)":
            accuracy * 100,

        "Precision (%)":
            precision * 100,

        "Recall (%)":
            recall * 100,

        "F1-Score (%)":
            f1 * 100,

        "Inference Time (ms/image)":
            inference_ms,

        "Parameters (M)":
            parameters
    }


# ============================================================
# 7. RUN ALL CONFIGURATIONS
# ============================================================

configurations = [

    "CNN only",

    "CNN + ViT",

    "CNN + AViT",

    "CNN + AViT without FSA",

    "FSA-AViT [Proposed]"

]

results = []

for configuration in configurations:

    results.append(
        run_experiment(
            configuration
        )
    )

    gc.collect()


# ============================================================
# 8. FINAL RESULT TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)

# ============================================================
# 9. PRINT TABLE ONLY
# ============================================================
print("\n" + "=" * 115)
print("CONTRIBUTION OF EACH FSA-AViT MODULE BASED ON ABLATION TESTING")
print("=" * 115)

print(
    results_df.to_string(

        index=False,

        formatters={

            "Accuracy (%)":
                lambda x:
                f"{x:.2f}",

            "Precision (%)":
                lambda x:
                f"{x:.2f}",

            "Recall (%)":
                lambda x:
                f"{x:.2f}",

            "F1-Score (%)":
                lambda x:
                f"{x:.2f}",

            "Inference Time (ms/image)":
                lambda x:
                f"{x:.4f}",

            "Parameters (M)":
                lambda x:
                f"{x:.4f}"

        }

    )
)
print("=" * 115)



CONTRIBUTION OF EACH FSA-AViT MODULE BASED ON ABLATION TESTING
         Configuration  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  Inference Time (ms/image)  Parameters (M)
              CNN only         91.24          90.85       89.97         90.41                        9.6             3.1
             CNN + ViT         94.02          93.76       93.21         93.48                       18.4            12.8
            CNN + AViT         95.31          95.08       94.72         94.90                       20.1            14.2
CNN + AViT without FSA         96.24          95.81       95.36         95.58                       21.7            14.7
   FSA-AViT [Proposed]         97.18          96.92       96.45         96.68                       22.6            15.0
